# Notebook 1: Descripción y calidad del dataset


## 1. Origen y carga de los datos
El dataset proviene de la Tarea 3, que identifica repositorios que utilizan GitHub Agentic Workflows y extrae sus archivos `.md`. Los datos se encuentran publicados en Hugging Face en el repositorio `EloyPradoMora/BitacoraDeGH-AW`.

Este notebook utiliza Pandas para conectarse directamente a Hugging Face usando el prefijo `hf://` y cargar las tablas en formato Parquet, sin necesidad de usar el token, ya que el repositorio es público.


In [ ]:
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

base_url = 'hf://datasets/EloyPradoMora/BitacoraDeGH-AW'

print('Cargando repositories...')
try:
    repo_df = pd.read_parquet(f'{base_url}/repositories.parquet')
except Exception as e:
    print(f'Error cargando repositories: {e}')
    repo_df = pd.DataFrame()

print('Cargando workflows...')
try:
    wf_df = pd.read_parquet(f'{base_url}/workflows.parquet')
except Exception as e:
    print(f'Error cargando workflows: {e}')
    wf_df = pd.DataFrame()

print('Cargando workflow attributes...')
try:
    attr_df = pd.read_parquet(f'{base_url}/workflow_attributes.parquet')
except Exception as e:
    print(f'Advertencia: workflow_attributes.parquet no se encontró en HF. Creando un DataFrame vacío temporal. {e}')
    attr_df = pd.DataFrame(columns=['workflow_id', 'key', 'value'])

print('¡Carga completada!')


Cargando repositories...


Cargando workflows...


Cargando workflow attributes...
Advertencia: workflow_attributes.parquet no se encontró en HF. Creando un DataFrame vacío temporal. datasets/EloyPradoMora/BitacoraDeGH-AW/workflow_attributes.parquet
¡Carga completada!


### ¿Qué representa cada tabla?
- **repositories**: Cada fila representa un repositorio único en GitHub que utiliza GHAW (identificado por la presencia conjunta de un archivo `.md` y un `.lock.yml`). Contiene metadatos del repositorio como estrellas, lenguaje, etc.
- **workflows**: Cada fila representa un archivo `.md` específico dentro del directorio `.github/workflows/` de un repositorio de GHAW. Contiene el nombre del archivo y el cuerpo de texto en Markdown.
- **workflow_attributes**: Cada fila representa un único par clave-valor del *frontmatter* YAML encontrado en los archivos Markdown (modelo EAV).


## 2. Descripción de las tablas y sus relaciones


In [ ]:
def describir_tabla(df, nombre):
    print(f'\n--- Tabla: {nombre} ---')
    print(f'Filas: {len(df):,}')
    print(f'Columnas: {len(df.columns)}')
    print('\nTipos de datos:')
    print(df.dtypes)

describir_tabla(repo_df, 'repositories')
describir_tabla(wf_df, 'workflows')
describir_tabla(attr_df, 'workflow_attributes')



--- Tabla: repositories ---
Filas: 60
Columnas: 36

Tipos de datos:
id                     int64
repo_id                  str
isFork                  bool
commits                int64
branches               int64
releases               int64
forks                  int64
mainLanguage             str
defaultBranch            str
license                  str
homepage                 str
watchers               int64
stargazers             int64
contributors         float64
size                   int64
createdAt                str
pushedAt                 str
updatedAt                str
totalIssues            int64
openIssues             int64
totalPullRequests      int64
openPullRequests       int64
blankLines           float64
codeLines            float64
commentLines         float64
metrics                  str
lastCommit               str
lastCommitSHA            str
hasWiki                 bool
isArchived              bool
isDisabled              bool
isLocked                bool
lan

### Claves y Relaciones
- **repositories**: PK = `repo_id`
- **workflows**: PK = `workflow_id`, FK = `repo_id` (hacia `repositories`)
- **workflow_attributes**: FK = `workflow_id` (hacia `workflows`)

### Conteo de elementos únicos


In [3]:
repos_unicos = repo_df['repo_id'].nunique() if 'repo_id' in repo_df.columns else 0
archivos_unicos = wf_df['workflow_id'].nunique() if 'workflow_id' in wf_df.columns else 0

print(f'Cantidad de repositorios únicos: {repos_unicos:,} (Total de filas en tabla: {len(repo_df):,})')
print(f'Cantidad de archivos Markdown únicos: {archivos_unicos:,} (Total de filas en tabla: {len(wf_df):,})')


Cantidad de repositorios únicos: 60 (Total de filas en tabla: 60)
Cantidad de archivos Markdown únicos: 153 (Total de filas en tabla: 153)


## 3. Revisión de calidad


In [4]:
print('--- Revisión de Nulos ---')
print('Nulos en repositories:\n', repo_df.isnull().sum())
print('\nNulos en workflows:\n', wf_df.isnull().sum())
print('\nNulos en attributes:\n', attr_df.isnull().sum())

print('\n--- Revisión de Duplicados ---')
print(f'Filas duplicadas en repositories: {repo_df.duplicated().sum()}')
print(f'Filas duplicadas en workflows: {wf_df.duplicated().sum()}')
print(f'Filas duplicadas en attributes: {attr_df.duplicated().sum()}')

print('\n--- Revisión de Claves Primarias (PK) ---')
if 'repo_id' in repo_df.columns:
    print(f'PKs repetidas en repositories (repo_id): {repo_df["repo_id"].duplicated().sum()}')
if 'workflow_id' in wf_df.columns:
    print(f'PKs repetidas en workflows (workflow_id): {wf_df["workflow_id"].duplicated().sum()}')

print('\n--- Revisión de Integridad Referencial (FK) ---')
if 'repo_id' in repo_df.columns and 'repo_id' in wf_df.columns:
    repo_ids_validos = set(repo_df['repo_id'])
    fk_invalidas_wf = wf_df[~wf_df['repo_id'].isin(repo_ids_validos)]
    print(f'Archivos asociados a un repositorio inexistente: {len(fk_invalidas_wf)}')

if 'workflow_id' in wf_df.columns and 'workflow_id' in attr_df.columns:
    wf_ids_validos = set(wf_df['workflow_id'])
    fk_invalidas_attr = attr_df[~attr_df['workflow_id'].isin(wf_ids_validos)]
    print(f'Atributos asociados a un archivo inexistente: {len(fk_invalidas_attr)}')


--- Revisión de Nulos ---
Nulos en repositories:
 id                    0
repo_id               0
isFork                0
commits               0
branches              0
releases              0
forks                 0
mainLanguage          0
defaultBranch         0
license               6
homepage             27
watchers              0
stargazers            0
contributors          0
size                  0
createdAt             0
pushedAt              0
updatedAt             0
totalIssues           0
openIssues            0
totalPullRequests     0
openPullRequests      0
blankLines            0
codeLines             0
commentLines          0
metrics               0
lastCommit            0
lastCommitSHA         0
hasWiki               0
isArchived            0
isDisabled            0
isLocked              0
languages             0
labels                0
topics               21
uses_gh_aw            0
dtype: int64

Nulos en workflows:
 workflow_id    0
repo_id        0
filename       0


## 4. Tratamiento de los problemas encontrados
A partir del análisis de calidad anterior, vemos si hay problemas. Si hay valores nulos en el cuerpo (`body`) de los workflows, podríamos rellenarlos con un string vacío o dejarlos como nulos. Los nulos en características opcionales de los repositorios (como *description* o *language*) son válidos y no requieren imputación porque simplemente reflejan que el repositorio en GitHub no completó esa información. Las claves primarias y foráneas están íntegras, y los tipos de datos están en su mayoría en formatos adecuados (`object`/`string` y numéricos).


In [ ]:
import os

os.makedirs('data/processed', exist_ok=True)

# Ejemplo de limpieza básica donde aseguramos que los strings no sean puramente nulos.
if 'body' in wf_df.columns:
    wf_df['body'] = wf_df['body'].fillna('')

print('Guardando tablas procesadas en eda/data/processed/ ...')
repo_df.to_parquet('data/processed/repositories.parquet', index=False)
wf_df.to_parquet('data/processed/workflows.parquet', index=False)
attr_df.to_parquet('data/processed/workflow_attributes.parquet', index=False)

print('¡Archivos guardados correctamente!')


Guardando tablas procesadas en eda/data/processed/ ...


¡Archivos guardados correctamente!
